# Bounded selectors and ES256

Numeric range selectors on Ed25519 major 2 and P-256 major 3, plus integral-float refusals and accepted int/fractional replacements.

Open this file in JupyterLab or VS Code with the **Deno** kernel. From a repository checkout, install Deno 2.5.6, Python 3.13.9 and uv 0.9.9 (the `.tool-versions` pins), run `uv sync --frozen` and `uv run --frozen deno jupyter --install`, then select Deno and Run All. Any Jupyter-capable editor can use that installed kernel. See [launch and automated checks](README.md).

The saved notebook installs exact published packages. `pnpm notebooks:local` checks the packed candidate; `pnpm notebooks:published` runs these cells unchanged against npm. Fixed times describe a demonstration, not current credentials. Keys and signatures are generated, never golden output. Host authorization, replay and revocation stay outside this notebook.


In [1]:
import signerMetadata from "npm:@bounded-authority-protocol/signer@2.0.0/package.json" with { type: "json" };
import verifierMetadata from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
if (signerMetadata.version !== "2.0.0" || verifierMetadata.version !== "2.0.0") throw new Error("package version mismatch");
console.log("exact signer and verifier releases loaded");

exact signer and verifier releases loaded


In [2]:
import { Buffer } from "node:buffer";
import { generateKeyPairSync, sign } from "node:crypto";
import { jwkFromPublicKey, thumbprint, v3 } from "npm:@bounded-authority-protocol/verifier@2.0.0";
import type { Es256KeyHandle, KeyHandle, KeyRole } from "npm:@bounded-authority-protocol/signer@2.0.0";

const fromBase64url = (text: string): Uint8Array => new Uint8Array(Buffer.from(text, "base64url"));

/** An Ed25519 handle over a fresh in-process key (contract-majors 1 and 2). */
function testKeyHandle(keyId: string, role: KeyRole = "holder"): KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ed25519");
  const raw = fromBase64url(publicKey.export({ format: "jwk" }).x!);
  return {
    sign: (message) => new Uint8Array(sign(null, message, privateKey)),
    publicKey: () => raw,
    thumbprint: () => thumbprint(jwkFromPublicKey(raw)),
    keyIdentity: () => ({ keyId, publicKey: raw }),
    signingIdentity: () => ({ role, keyId, publicKey: raw }),
  };
}

/** An ES256 handle over a fresh in-process P-256 key (contract-major 3). */
function testEs256KeyHandle(keyId: string, role: KeyRole = "holder"): Es256KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ec", { namedCurve: "P-256" });
  const { x, y } = publicKey.export({ format: "jwk" }) as { x: string; y: string };
  const raw = new Uint8Array([0x04, ...fromBase64url(x), ...fromBase64url(y)]);
  return {
    sign: (message) => new Uint8Array(sign("sha256", message, { key: privateKey, dsaEncoding: "ieee-p1363" })),
    publicKey: () => raw,
    thumbprint: () => v3.thumbprint({ crv: "P-256", kty: "EC", x, y }),
    keyIdentity: () => ({ keyId, publicKey: raw }),
    signingIdentity: () => ({ role, keyId, publicKey: raw }),
  };
}
console.log("demonstration key constructors ready");

demonstration key constructors ready


In [3]:
import { signV2Grant, signV2Report } from "npm:@bounded-authority-protocol/signer@2.0.0";
import { v2, type Tagged } from "npm:@bounded-authority-protocol/verifier@2.0.0";

{


const home = testKeyHandle("home-issuer-1", "issuer");
const assistant = testKeyHandle("assistant-key-1");

const grant = await signV2Grant({
  issuer: "https://home.example", grantId: "urn:example:grant:thermostat",
  audiences: ["https://thermostat.example"],
  issuedAt: 1_767_225_600, notBefore: 1_767_225_600, expiresAt: 1_767_312_000,
  holderThumbprint: await assistant.thumbprint(),
  operations: [{ name: "set_temperature", selectors: [
    { kind: "gte", path: ["celsius"], value: { t: "float", v: 18.5 } },
    { kind: "lte", path: ["celsius"], value: { t: "float", v: 24.5 } },
  ] }],
}, home);
if (!grant.ok) throw new Error(`signV2Grant: ${grant.error}`);
const grantCompact = grant.value.grant;

async function setTemperature(celsius: number): Promise<boolean> {
  const request = {
    method: "PUT", targetUri: "https://thermostat.example/setpoint",
    invocationId: "3b9e1c52-8d4f-4a1e-b7c2-5e6f7a8b9c0d", operation: "set_temperature",
    castArguments: { t: "object", v: new Map<string, Tagged>([["celsius", { t: "float", v: celsius }]]) } satisfies Tagged,
  };
  const envelope = await signV2Report({ grantCompact, ...request }, assistant, { issuedAt: 1_767_225_660 });
  if (!envelope.ok) throw new Error(`signV2Report: ${envelope.error}`);
  return v2.checkEnvelope(envelope.value.grant, envelope.value.proof, {
    ...request,
    trustedIssuer: { keyId: "home-issuer-1", publicKey: await home.publicKey() },
    issuer: "https://home.example", audience: "https://thermostat.example",
    evaluationTime: 1_767_225_700, clockSkew: 60, proofMaxAge: 300, nonce: { kind: "not_required" },
  }).ok;
}

if (!await setTemperature(21.5) || await setTemperature(30.5)) throw new Error("range selector verdict");
console.log(`21.5 °C verifies: ${await setTemperature(21.5)}`);
console.log(`30.5 °C verifies: ${await setTemperature(30.5)}`);
}


21.5 °C verifies: true
30.5 °C verifies: false


In [4]:
import { signV3Grant, signV3Report } from "npm:@bounded-authority-protocol/signer@2.0.0";
import { strUtf8, v3, type Tagged } from "npm:@bounded-authority-protocol/verifier@2.0.0";

{


const platform = testEs256KeyHandle("platform-issuer-1", "issuer");
const deployBot = testEs256KeyHandle("deploy-bot-1");

const grant = await signV3Grant({
  issuer: "https://platform.example", grantId: "urn:example:grant:scale-web",
  audiences: ["https://deploy.example"],
  issuedAt: 1_767_225_600, notBefore: 1_767_225_600, expiresAt: 1_767_227_400,
  holderThumbprint: await deployBot.thumbprint(),
  operations: [{ name: "scale_service", selectors: [
    { kind: "equals", path: ["service"], value: { t: "string", v: strUtf8("web") } },
    { kind: "lte", path: ["replicas"], value: { t: "int", v: 10 } },
  ] }],
}, platform);
if (!grant.ok) throw new Error(`signV3Grant: ${grant.error}`);

const request = {
  method: "POST", targetUri: "https://deploy.example/scale",
  invocationId: "6a1f0d3c-2b4e-4f5a-8c6d-7e8f9a0b1c2d", operation: "scale_service",
  castArguments: { t: "object", v: new Map<string, Tagged>([
    ["service", { t: "string", v: strUtf8("web") }],
    ["replicas", { t: "int", v: 4 }],
  ]) } satisfies Tagged,
};
const envelope = await signV3Report({ grantCompact: grant.value.grant, ...request }, deployBot, { issuedAt: 1_767_225_660 });
if (!envelope.ok) throw new Error(`signV3Report: ${envelope.error}`);

const facts = v3.checkEnvelope(envelope.value.grant, envelope.value.proof, {
  ...request,
  trustedIssuer: { keyId: "platform-issuer-1", publicKey: await platform.publicKey() },
  issuer: "https://platform.example", audience: "https://deploy.example",
  evaluationTime: 1_767_225_700, clockSkew: 60, proofMaxAge: 300, nonce: { kind: "not_required" },
});
if (!facts.ok) throw new Error("v3 envelope refused");
console.log(`v3 envelope verifies: ${facts.ok}`);
}


v3 envelope verifies: true


In [5]:
import { signGrant, signV2Grant, signV3Grant } from "npm:@bounded-authority-protocol/signer@2.0.0";
import type { Tagged } from "npm:@bounded-authority-protocol/verifier@2.0.0";

{


const issuer = testKeyHandle("backup-issuer-1", "issuer");
const es256Issuer = testEs256KeyHandle("backup-issuer-2", "issuer");
const agent = testKeyHandle("backup-agent-1");

const base = {
  issuer: "https://backup.example", grantId: "urn:example:grant:retention",
  audiences: ["https://snapshots.example"],
  issuedAt: 1_767_225_600, notBefore: 1_767_225_600, expiresAt: 1_767_312_000,
  holderThumbprint: await agent.thumbprint(),
};
const days = (value: Tagged) => ({ path: ["days"], value });
const results = {
  "signGrant, equals 30.0": await signGrant({ ...base, operations: [{ name: "set_retention", selectors: [{ kind: "equals", ...days({ t: "float", v: 30.0 }) }] }] }, issuer),
  "signV2Grant, lte 30.0": await signV2Grant({ ...base, operations: [{ name: "set_retention", selectors: [{ kind: "lte", ...days({ t: "float", v: 30.0 }) }] }] }, issuer),
  "signV3Grant, lte 30.0": await signV3Grant({ ...base, operations: [{ name: "set_retention", selectors: [{ kind: "lte", ...days({ t: "float", v: 30.0 }) }] }] }, es256Issuer),
  "signV3Grant, lte 30 (int)": await signV3Grant({ ...base, operations: [{ name: "set_retention", selectors: [{ kind: "lte", ...days({ t: "int", v: 30 }) }] }] }, es256Issuer),
  "signV3Grant, lte 30.5": await signV3Grant({ ...base, operations: [{ name: "set_retention", selectors: [{ kind: "lte", ...days({ t: "float", v: 30.5 }) }] }] }, es256Issuer),
};
for (const [label, result] of Object.entries(results)) console.log(`${label}: ${result.ok ? "signed" : result.error}`);
if (Object.entries(results).some(([name, result]) => name.includes("30.0") ? result.ok || result.error !== "producer_error" : !result.ok)) throw new Error("integral selector refusal");
}


signGrant, equals 30.0: producer_error
signV2Grant, lte 30.0: producer_error
signV3Grant, lte 30.0: producer_error
signV3Grant, lte 30 (int): signed
signV3Grant, lte 30.5: signed


## Cleanup

These cells keep keys and compacts only in the kernel. Shut down the kernel to discard them. No private material is persisted. The runner shuts down the kernel and removes its consumer directory, package caches and temporary kernel registration, and fails on files left in the notebook working directory.

Continue with the [guides](../guides/getting-started.md).
